In [45]:
import polars as pl
from pathlib import Path
from datetime import datetime

In [2]:
BASE_PATH = Path.cwd().parent / "titanic-competition"
TRAIN_PATH = BASE_PATH / "train.csv"

In [3]:
# As we saw there are two approaches : lazy and eager
# One loads the data into ram(but still better than pandas), and stores the workflow to follow when loading data from path.

eager_df = pl.read_csv(TRAIN_PATH)
lazy_df = pl.scan_csv(TRAIN_PATH)

In [4]:
# Now let's peek at the head 
eager_df.head()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


In [5]:
# Look at the cleanliness and format .
# Compared to pandas a much more descriptive and informative peeking
# Also looking closely above , it first loads the shape of the slice and then loads the data.

In [6]:
# Now instead of info here we have scheme which showcases all the colnames and it's dtype immediately each in a tuple.
eager_df.schema

Schema([('PassengerId', Int64),
        ('Survived', Int64),
        ('Pclass', Int64),
        ('Name', String),
        ('Sex', String),
        ('Age', Float64),
        ('SibSp', Int64),
        ('Parch', Int64),
        ('Ticket', String),
        ('Fare', Float64),
        ('Cabin', String),
        ('Embarked', String)])

In [7]:
# Many functions have same name compared to pandas .
# The creators of Polars wanted the transition to feel familiar.
# The real difference begins when we start manipulating the data.

eager_df.head()
eager_df.shape
eager_df.columns
eager_df.dtypes


[Int64,
 Int64,
 Int64,
 String,
 String,
 Float64,
 Int64,
 Int64,
 String,
 Float64,
 String,
 String]

In [8]:
eager_df.sample(5)

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
661,1,1,"""Frauenthal, Dr. Henry William""","""male""",50.0,2,0,"""PC 17611""",133.65,null,"""S"""
601,1,2,"""Jacobsohn, Mrs. Sidney Samuel …","""female""",24.0,2,1,"""243847""",27.0,null,"""S"""
336,0,3,"""Denkoff, Mr. Mitto""","""male""",null,0,0,"""349225""",7.8958,null,"""S"""
685,0,2,"""Brown, Mr. Thomas William Solo…","""male""",60.0,1,1,"""29750""",39.0,null,"""S"""
791,0,3,"""Keane, Mr. Andrew ""Andy""""","""male""",null,0,0,"""12460""",7.75,null,"""Q"""


In [9]:
# We cannnot do boolean masking like pandas and we strictly need to follow filter function for effecient optimized code.

In [10]:
# With pl.col() we can create expression to filter out effeciently.
# NOTE :  This is not a boolean array as eager_df[eager_df['Sex'=='female']] but an expression
type(pl.col("Sex")) # We can clearly see that this is a type of expression 

polars.expr.expr.Expr

In [11]:
female_survivor = eager_df.filter(
                    pl.col("Sex") == "female",
                    pl.col("Survived") == 1
)

In [12]:
female_survivor

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
9,1,3,"""Johnson, Mrs. Oscar W (Elisabe…","""female""",27.0,0,2,"""347742""",11.1333,null,"""S"""
10,1,2,"""Nasser, Mrs. Nicholas (Adele A…","""female""",14.0,1,0,"""237736""",30.0708,null,"""C"""
…,…,…,…,…,…,…,…,…,…,…,…
875,1,2,"""Abelson, Mrs. Samuel (Hannah W…","""female""",28.0,1,0,"""P/PP 3381""",24.0,null,"""C"""
876,1,3,"""Najib, Miss. Adele Kiamie ""Jan…","""female""",15.0,0,0,"""2667""",7.225,null,"""C"""
880,1,1,"""Potter, Mrs. Thomas Jr (Lily A…","""female""",56.0,0,1,"""11767""",83.1583,"""C50""","""C"""


In [13]:
# Polars isn't built around DataFrames.It's built around expressions.

In [14]:
# Now we will dealt with with_columns()
# Why we need this ? - To Chain multiple expressions into a single expression so that we can create multiple new columns at one go .
# Let's take for example  : 
df = eager_df.with_columns(
    (
        pl.col("SibSp") +
        pl.col("Parch") +
        1
    ).alias("FamilySize"), # We add alias to give the expression result a destiny , without this is there is no home for the result. 
                           # We can also speicfy the same name as prexisting column as well to modify it's behaviour.

    (
        pl.col("Age") < 18
    ).alias("IsChild"),

    (
        pl.col("Fare") > 100
    ).alias("HighFare")
)
# It can literally compute everything at one go whereas in pandas this would be too hectic , to create new features one by one and would be completely ineffecient.
# Why ? Cause in pandas we would have had to load each column once then add each value of other columns and so on ....

# But with_columns let's us make new columns instantly with effeciency; 

In [15]:
df.head()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,IsChild,HighFare
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,i64,bool,bool
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S""",2,false,false
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",2,false,false
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",1,false,false
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S""",2,false,false
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S""",1,false,false


In [16]:
# We can also modify the current columns at one go too.
# For example : 
new_age = df.with_columns(
    (pl.col("Age") + 1).alias("Age")
)

In [17]:
new_age.head() # COMPARE this with the dataframe above to check whether there was an actual change in tha "Age" column

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,IsChild,HighFare
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,i64,bool,bool
1,0,3,"""Braund, Mr. Owen Harris""","""male""",23.0,1,0,"""A/5 21171""",7.25,null,"""S""",2,false,false
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",39.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",2,false,false
3,1,3,"""Heikkinen, Miss. Laina""","""female""",27.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",1,false,false
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",36.0,1,0,"""113803""",53.1,"""C123""","""S""",2,false,false
5,0,3,"""Allen, Mr. William Henry""","""male""",36.0,0,0,"""373450""",8.05,null,"""S""",1,false,false


In [18]:
# Now when we have two or more expression then what happens is that both expression use the original age group simultaenously
# Polars guarantees:Every expression in one with_columns() sees the same original snapshot of the DataFrame.
# take this below example : 
eager_df.with_columns(
    (pl.col("Age") + 1).alias("Age"),
    (pl.col("Age") > 18).alias("Adult")
)

# Now what this it does is  : 
# 1) Give the original "Age" to both the equations
# 2) Exp 1 updated the "Age" by adding 1 but as the Exp 2 has the raw "Age" as well the updated "Age " is never passed to form the "Adult"

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Adult
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,bool
1,0,3,"""Braund, Mr. Owen Harris""","""male""",23.0,1,0,"""A/5 21171""",7.25,null,"""S""",true
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",39.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",true
3,1,3,"""Heikkinen, Miss. Laina""","""female""",27.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",true
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",36.0,1,0,"""113803""",53.1,"""C123""","""S""",true
5,0,3,"""Allen, Mr. William Henry""","""male""",36.0,0,0,"""373450""",8.05,null,"""S""",true
…,…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,"""Montvila, Rev. Juozas""","""male""",28.0,0,0,"""211536""",13.0,null,"""S""",true
888,1,1,"""Graham, Miss. Margaret Edith""","""female""",20.0,0,0,"""112053""",30.0,"""B42""","""S""",true
889,0,3,"""Johnston, Miss. Catherine Hele…","""female""",null,1,2,"""W./C. 6607""",23.45,null,"""S""",null


In [19]:
# If we really wanted to bypass this we can use 
eager_df.with_columns(
    (pl.col("Age") + 1).alias("Age")
).with_columns(
        (pl.col("Age") > 18).alias("Adult")
)

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Adult
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,bool
1,0,3,"""Braund, Mr. Owen Harris""","""male""",23.0,1,0,"""A/5 21171""",7.25,null,"""S""",true
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",39.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",true
3,1,3,"""Heikkinen, Miss. Laina""","""female""",27.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",true
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",36.0,1,0,"""113803""",53.1,"""C123""","""S""",true
5,0,3,"""Allen, Mr. William Henry""","""male""",36.0,0,0,"""373450""",8.05,null,"""S""",true
…,…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,"""Montvila, Rev. Juozas""","""male""",28.0,0,0,"""211536""",13.0,null,"""S""",true
888,1,1,"""Graham, Miss. Margaret Edith""","""female""",20.0,0,0,"""112053""",30.0,"""B42""","""S""",true
889,0,3,"""Johnston, Miss. Catherine Hele…","""female""",null,1,2,"""W./C. 6607""",23.45,null,"""S""",null


In [20]:
# Feature Engineering using with_columns() : 
"""
FamilySize = SibSp + Parch + 1
IsChild = Age < 18
FarePerPerson = Fare / FamilySize
"""

df = eager_df.with_columns(
    (pl.col("SibSp") + pl.col("Parch") + 1).alias("FamilySize"),
    (pl.col("Age") < 18).alias("IsChild")).with_columns(
        (pl.col("Fare") / pl.col("FamilySize")).alias("FarePerPerson")
    )


In [21]:
df

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,IsChild,FarePerPerson
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,i64,bool,f64
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S""",2,false,3.625
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",2,false,35.64165
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",1,false,7.925
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S""",2,false,26.55
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S""",1,false,8.05
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,"""Montvila, Rev. Juozas""","""male""",27.0,0,0,"""211536""",13.0,null,"""S""",1,false,13.0
888,1,1,"""Graham, Miss. Margaret Edith""","""female""",19.0,0,0,"""112053""",30.0,"""B42""","""S""",1,false,30.0
889,0,3,"""Johnston, Miss. Catherine Hele…","""female""",null,1,2,"""W./C. 6607""",23.45,null,"""S""",4,null,5.8625


In [22]:
# A better version would be to substitute the equation itself instead of nesting the with_columns() again
df = eager_df.with_columns(
    (pl.col("SibSp") + pl.col("Parch") + 1).alias("FamilySize"),

    (pl.col("Age") < 18).alias("IsChild"),

    (
        pl.col("Fare") /
        (pl.col("SibSp") + pl.col("Parch") + 1)
    ).alias("FarePerPerson")
)

In [23]:
# Q :  Find the number of survivors 
n_survivors = eager_df.select("Survived").filter(pl.col("Survived")==1).sum()
n_survivors = eager_df.select("Survived").sum()
n_survivors = (
    eager_df
    .filter(pl.col("Survived") == 1)
    .height
)
# Mutiple ways to do the same thing but the second one is the most optimum followed by 3 and 2


In [24]:
# Now grouping by using a categorical column
eager_df.group_by("Pclass").agg(
    pl.col("Fare").mean()
)

Pclass,Fare
i64,f64
3,13.67555
1,84.154687
2,20.662183


In [25]:
# In behind scene : 
""" 
You never stored : 

Class 1 DataFrame
        ↓   
Compute Mean

You simply maintained : 

Running Sum
Running Count

This is called streaming aggregation.

Modern query engines—including Polars—often use strategies very similar to this because they avoid materializing lots of intermediate data.
"""

' \nYou never stored : \n\nClass 1 DataFrame\n        ↓   \nCompute Mean\n\nYou simply maintained : \n\nRunning Sum\nRunning Count\n\nThis is called streaming aggregation.\n\nModern query engines—including Polars—often use strategies very similar to this because they avoid materializing lots of intermediate data.\n'

In [26]:
eager_df

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""
…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,"""Montvila, Rev. Juozas""","""male""",27.0,0,0,"""211536""",13.0,null,"""S"""
888,1,1,"""Graham, Miss. Margaret Edith""","""female""",19.0,0,0,"""112053""",30.0,"""B42""","""S"""
889,0,3,"""Johnston, Miss. Catherine Hele…","""female""",null,1,2,"""W./C. 6607""",23.45,null,"""S"""


In [27]:
# Now we can go further and add more aggregate functions as well as we already are acessing the PClass that is  : 
eager_df.group_by("Pclass").agg(
    pl.col("Fare").mean(),
    pl.col("Fare").max().alias("Fare Max"), # Use alias cause column name cannot be same if we are using multiple expression on a single column
    pl.col("Age").mean(),
    pl.len(),# pl.col("PassengerId").count(),
    pl.col("Survived").sum()
)

Pclass,Fare,Fare Max,Age,len,Survived
i64,f64,f64,f64,u32,i64
2,20.662183,73.5,29.87763,184,87
1,84.154687,512.3292,38.233441,216,136
3,13.67555,69.55,25.14062,491,119


In [28]:
# Our engine can do  : 
""" 
Read Row - > Update Mean - > Update Max - > Update Count - >Update Sum - > Next Row
"""

' \nRead Row - > Update Mean - > Update Max - > Update Count - >Update Sum - > Next Row\n'

In [29]:
# Now what if we dont want the summary but each row intact with also the aggregate then we can use :  Window functions 
# What they do is : Instead of looking up the complete table they keep track of the aggregate functions and attach the new value over to original dataframe
# Take for example :
eager_df.with_columns(
    pl.col("Fare")
      .mean()
      .over("Pclass")
      .alias("ClassAverageFare")
).head(10)

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,ClassAverageFare
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,f64
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S""",13.67555
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",84.154687
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",13.67555
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S""",84.154687
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S""",13.67555
6,0,3,"""Moran, Mr. James""","""male""",null,0,0,"""330877""",8.4583,null,"""Q""",13.67555
7,0,1,"""McCarthy, Mr. Timothy J""","""male""",54.0,0,0,"""17463""",51.8625,"""E46""","""S""",84.154687
8,0,3,"""Palsson, Master. Gosta Leonard""","""male""",2.0,3,1,"""349909""",21.075,null,"""S""",13.67555
9,1,3,"""Johnson, Mrs. Oscar W (Elisabe…","""female""",27.0,0,2,"""347742""",11.1333,null,"""S""",13.67555


In [30]:
# Now at the first glance we come to know that  : 
# This is not a summary any more where every other columns disappear
# We calculate the complete aggregate and simply attach it back to the original table for each  class 

In [31]:
# We can calculate the differnce between a fare of a person in given class and the average fare of the pclass
eager_df.with_columns(

    pl.col("Fare")
      .mean()
      .over("Pclass")
      .alias("ClassAvg"),

    (
        pl.col("Fare") # The fare values
        -
        pl.col("Fare") # The complete mean over each Pclass 
            .mean()
            .over("Pclass")
    ).alias("Difference")
).select("Pclass","Fare","ClassAvg","Difference")
# Why not just use the ClassAvg again for second column,you say ? 
# That would simply lead to ineffecient code as we already have first the workplan and the no cols to use within a single with_columns()

Pclass,Fare,ClassAvg,Difference
i64,f64,f64,f64
3,7.25,13.67555,-6.42555
1,71.2833,84.154687,-12.871387
3,7.925,13.67555,-5.75055
1,53.1,84.154687,-31.054687
3,8.05,13.67555,-5.62555
…,…,…,…
2,13.0,20.662183,-7.662183
1,30.0,84.154687,-54.154687
3,23.45,13.67555,9.77445


| Operation          | Number of Rows                          |
| ------------------ | --------------------------------------- |
| `group_by().agg()` | Usually decreases                       |
| `filter()`         | Decreases                               |
| `select()`         | Same rows, fewer columns                |
| `with_columns()`   | Same rows, more/updated columns         |
| `.over()`          | Same rows, but group-aware computations |


In [32]:
# Exercise : 
""" 
ClassAverageFare → average fare of each passenger's class.
FareDifference → passenger's fare - class average fare.
AboveClassAverage → True if the passenger paid more than the average fare of their class.
"""
eager_df.with_columns(
                        pl.col("Fare").mean().over("Pclass").alias("ClassAverageFare"),
                        (pl.col("Fare") - pl.col("Fare").mean().over("Pclass")).alias("FareDifference"),
                        (pl.col("Fare") > pl.col("Fare").mean().over("Pclass")).alias("AboveClassAverage")
).select("Fare","ClassAverageFare","FareDifference","AboveClassAverage")

# Actually when the with_columns() method will be runned : the term : pl.col("Fare").mean().over("Pclass")) will not be calculated again and again but smartly 
# manipulated in such a way that every operation is done once and there is no need to do the operations again and again

Fare,ClassAverageFare,FareDifference,AboveClassAverage
f64,f64,f64,bool
7.25,13.67555,-6.42555,false
71.2833,84.154687,-12.871387,false
7.925,13.67555,-5.75055,false
53.1,84.154687,-31.054687,false
8.05,13.67555,-5.62555,false
…,…,…,…
13.0,20.662183,-7.662183,false
30.0,84.154687,-54.154687,false
23.45,13.67555,9.77445,true


In [33]:
# Again we are more focused on lazy implementation.
# It is almost as if this is the mixture of pandas and sql.
# We are mostly worried about the expressions and optimizing the code .

In [43]:
# Joins : Are basically referencing a key column from both the columns and perform operations based on the columns .

Here is a quick-reference summary table of all Polars join types, their behaviors, and key requirements.

| Join Strategy (how) | Rows Kept | Right Columns Kept? | Missing Matches Filled With? | Key Column Requirements |
|---|---|---|---|---|
| "inner" | Matching keys only | Yes | N/A | Requires on or left_on/right_on |
| "left" | All Left rows | Yes | null | Requires on or left_on/right_on |
| "right" | All Right rows | Yes | null | Requires on or left_on/right_on |
| "full" | All Left and Right rows | Yes | null | Requires on or left_on/right_on |
| "semi" | Left rows that match Right | No | N/A | Requires on or left_on/right_on |
| "anti" | Left rows that do not match Right | No | N/A | Requires on or left_on/right_on |
| "cross" | Every Left row × Every Right row | Yes | N/A | Omit all on parameters |
| .join_asof() | Left rows matched to nearest key | Yes | null | Nearest key match (e.g., timestamps) |
| .join_where() | Matches satisfying custom conditions | Yes | N/A | Uses inequality expressions (>, <, etc.) |


In [46]:
# Artificial dataset : 

# 1. Passengers DataFrame (5 rows: contains duplicates for ID 3, and an unmatched ID 4)
passengers = pl.DataFrame({
    "PassengerId": [1, 2, 3, 3, 4], 
    "Name": ["John", "Alice", "Bob", "Robert", "Charlie"], 
    "Pclass": [3, 1, 3, 1, 2],
    "Timestamp": [
        datetime(2026, 7, 25, 10, 0),
        datetime(2026, 7, 25, 10, 15),
        datetime(2026, 7, 25, 10, 30),
        datetime(2026, 7, 25, 10, 30),
        datetime(2026, 7, 25, 10, 45)
    ]
}).sort("Timestamp")  # Mandatory for join_asof

# 2. Tickets DataFrame (4 rows: contains an unmatched ID 5)
tickets = pl.DataFrame({
    "PassengerId": [1, 2, 3, 5], 
    "Fare": [7.25, 71.28, 8.05, 50.00], 
    "Cabin": ["C85", "C123", None, "B45"],
    "Timestamp": [
        datetime(2026, 7, 25, 10, 2),   # 2 mins after John
        datetime(2026, 7, 25, 10, 14),  # 1 min before Alice
        datetime(2026, 7, 25, 10, 35),  # 5 mins after Bob
        datetime(2026, 7, 25, 10, 50)   # No matching passenger
    ]
}).sort("Timestamp")  # Mandatory for join_asof



In [ ]:
# Left Join
# All the other joins are similar the main thing that changes is the how condition

passengers.join(
    tickets,
    on="PassengerId",
    how="left"
)
# As we can clearly see that all the rows present in the left column is present ; and the rows which were not intersecting got replaced but null

PassengerId,Name,Pclass,Timestamp,Fare,Cabin,Timestamp_right
i64,str,i64,datetime[μs],f64,str,datetime[μs]
1,"""John""",3,2026-07-25 10:00:00,7.25,"""C85""",2026-07-25 10:02:00
2,"""Alice""",1,2026-07-25 10:15:00,71.28,"""C123""",2026-07-25 10:14:00
3,"""Bob""",3,2026-07-25 10:30:00,8.05,null,2026-07-25 10:35:00
3,"""Robert""",1,2026-07-25 10:30:00,8.05,null,2026-07-25 10:35:00
4,"""Charlie""",2,2026-07-25 10:45:00,null,null,null


In [ ]:
# Right Join

passengers.join(
    tickets,
    on="PassengerId",
    how="left"
)

PassengerId,Name,Pclass,Timestamp,Fare,Cabin,Timestamp_right
i64,str,i64,datetime[μs],f64,str,datetime[μs]
1,"""John""",3,2026-07-25 10:00:00,7.25,"""C85""",2026-07-25 10:02:00
2,"""Alice""",1,2026-07-25 10:15:00,71.28,"""C123""",2026-07-25 10:14:00
3,"""Bob""",3,2026-07-25 10:30:00,8.05,null,2026-07-25 10:35:00
3,"""Robert""",1,2026-07-25 10:30:00,8.05,null,2026-07-25 10:35:00
4,"""Charlie""",2,2026-07-25 10:45:00,null,null,null


In [49]:
# Inner Join (Intersection)

passengers.join(
    tickets,
    on="PassengerId",
    how="inner"
)

PassengerId,Name,Pclass,Timestamp,Fare,Cabin,Timestamp_right
i64,str,i64,datetime[μs],f64,str,datetime[μs]
1,"""John""",3,2026-07-25 10:00:00,7.25,"""C85""",2026-07-25 10:02:00
2,"""Alice""",1,2026-07-25 10:15:00,71.28,"""C123""",2026-07-25 10:14:00
3,"""Bob""",3,2026-07-25 10:30:00,8.05,null,2026-07-25 10:35:00
3,"""Robert""",1,2026-07-25 10:30:00,8.05,null,2026-07-25 10:35:00


In [51]:
# Full Join 

passengers.join(
    tickets,
    on="PassengerId",
    how="full"
)
# Union of Left and Right Join

PassengerId,Name,Pclass,Timestamp,PassengerId_right,Fare,Cabin,Timestamp_right
i64,str,i64,datetime[μs],i64,f64,str,datetime[μs]
1,"""John""",3,2026-07-25 10:00:00,1,7.25,"""C85""",2026-07-25 10:02:00
2,"""Alice""",1,2026-07-25 10:15:00,2,71.28,"""C123""",2026-07-25 10:14:00
3,"""Bob""",3,2026-07-25 10:30:00,3,8.05,null,2026-07-25 10:35:00
3,"""Robert""",1,2026-07-25 10:30:00,3,8.05,null,2026-07-25 10:35:00
4,"""Charlie""",2,2026-07-25 10:45:00,null,null,null,null
null,null,null,null,5,50.0,"""B45""",2026-07-25 10:50:00


In [39]:
# Joining columns with different name but same content from two diff tables

"""
passengers.join(
    tickets,
    left_on="PassengerId",
    right_on="pid", # We can specify the columns to join on if the column names are differnt but content is the same
    how="left"
)
"""

'\npassengers.join(\n    tickets,\n    left_on="PassengerId",\n    right_on="pid", # We can specify the columns to join on if the column names are differnt but content is the same\n    how="left"\n)\n'

In [41]:
# Multiple Join Keys :  When we have to join based on multiple column intersection
"""
df1.join(
    df2,
    on=["customer_id", "date"],
    how="left"
)
"""

'\ndf1.join(\n    df2,\n    on=["customer_id", "date"],\n    how="left"\n)\n'

In [52]:
# Real life example in our titanic dataset : 
class_stats = (
    eager_df
    .group_by("Pclass")
    .agg(
        pl.col("Fare").mean().alias("ClassAvgFare"),
        pl.col("Fare").max().alias("ClassMaxFare")
    )
)

In [53]:
class_stats

Pclass,ClassAvgFare,ClassMaxFare
i64,f64,f64
2,20.662183,73.5
3,13.67555,69.55
1,84.154687,512.3292


In [55]:
eager_df.join(
    class_stats,
    on="Pclass",
    how="left"
)
# We can use the grouped values to be appended to each of our original row and this way we can create a new column for analysis 

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,ClassAvgFare,ClassMaxFare
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,f64,f64
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S""",13.67555,69.55
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",84.154687,512.3292
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",13.67555,69.55
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S""",84.154687,512.3292
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S""",13.67555,69.55
…,…,…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,"""Montvila, Rev. Juozas""","""male""",27.0,0,0,"""211536""",13.0,null,"""S""",20.662183,73.5
888,1,1,"""Graham, Miss. Margaret Edith""","""female""",19.0,0,0,"""112053""",30.0,"""B42""","""S""",84.154687,512.3292
889,0,3,"""Johnston, Miss. Catherine Hele…","""female""",null,1,2,"""W./C. 6607""",23.45,null,"""S""",13.67555,69.55


In [56]:
# Joins are needed when :

# data comes from another table
# keys differ
# you need many columns from another dataset
# you're combining independently generated feature table